# LPQuant Baseline Suite

This notebook is the default research entry point for the current LP interval framework.

The workflow is deliberately standardized:

1. Run the baseline case matrix
2. Inspect pair-level comparison tables
3. Visualize the results with standard heatmaps and bar charts
4. Save the whole run as a timestamped experiment bundle
5. Drill into one pair and save that deep-dive as well


In [ ]:
import pandas as pd

from app.research import (
    BASELINE_CASES,
    build_pair_comparison_table,
    create_experiment_dir,
    plot_study_candidate_frontier,
    plot_summary_metric_bars,
    plot_summary_metric_heatmap,
    run_benchmark_suite,
    run_study,
    save_benchmark_suite,
    save_study_artifacts,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

len(BASELINE_CASES), [case.name for case in BASELINE_CASES[:4]]


## Run the baseline suite

The suite keeps going even if one pair cannot be loaded. Any failures are captured in `suite.failures`.


In [ ]:
suite = run_benchmark_suite(BASELINE_CASES)
suite.summary


In [ ]:
suite.failures


## Pair comparison table

This is the compact table to compare the top-ranked interval across pairs and scenarios.


In [ ]:
comparison_table = build_pair_comparison_table(suite.summary)
comparison_table


## Standard visual comparisons

These plots should stay consistent across experiments so the outputs are easy to compare run over run.


In [ ]:
plot_summary_metric_heatmap(
    suite.summary,
    "score",
    title="Top interval score by pair and scenario",
)


In [ ]:
plot_summary_metric_heatmap(
    suite.summary,
    "mean_fee_proxy_bps",
    title="Fee proxy by pair and scenario",
)


In [ ]:
plot_summary_metric_bars(
    suite.summary,
    "no_exit_rate",
    title="No-exit rate by pair and scenario",
)


In [ ]:
plot_summary_metric_bars(
    suite.summary,
    "downside_breach_rate",
    title="Downside breach rate by pair and scenario",
)


## Save the experiment bundle

This writes the entire benchmark run as a timestamped bundle under `research_runs/`, including suite metadata and per-case artifacts.


In [ ]:
run_dir = create_experiment_dir("research_runs", "baseline_suite")
saved_paths = save_benchmark_suite(
    suite,
    run_dir,
    label="baseline_suite",
    notes="Default baseline benchmark matrix from the notebook entry point.",
    tags=["baseline", "notebook", "pair-comparison"],
)
saved_paths


## Single-pair deep dive

Pick one case, rerun it, inspect the full frontier, and then save that focused study into the same run bundle.


In [ ]:
selected_case = next(case for case in BASELINE_CASES if case.name == "sui_usdc_4h_neutral_balanced")
study = run_study(selected_case.request)
study.rankings.head(20)


In [ ]:
plot_study_candidate_frontier(
    study.rankings.head(20),
    x="mean_in_range_pct",
    y="mean_fee_proxy_bps",
    title=f"Candidate frontier: {selected_case.name}",
)


In [ ]:
study.similar_windows[[
    "similarity_rank",
    "entry_time",
    "close",
    "trend_pct",
    "realized_vol_pct",
    "distance_to_sma_pct",
    "drawdown_pct",
    "rsi",
    "distance",
]].head(15)


In [ ]:
pair_focus = "SUI/USDC"
build_pair_comparison_table(suite.summary, pair=pair_focus)


In [ ]:
deep_dive_paths = save_study_artifacts(
    study,
    run_dir / "pair_deep_dives" / selected_case.name,
    label=selected_case.name,
    notes="Single-pair deep dive saved from the baseline notebook.",
    tags=["deep-dive", pair_focus.lower().replace("/", "_")],
)
deep_dive_paths
